# agentic-analytics: full eval suite on Kaggle (2×T4, vLLM, Qwen3.8-27B AWQ)

Runs the **30-business eval suite** (`aa eval --suite full`) with **Qwen3.8-27B, 4-bit (W4A16/AWQ-style)**, served by **vLLM inside this notebook** on Kaggle's free **2× Tesla T4**. No tunnel, no API key, no rate limits. The result is a results JSON in exactly the same schema as every other run, published as its own model row: **`vllm (Kaggle 2×T4) Qwen3.8-27B-AWQ`**.

> **Not yet run on Kaggle.** This notebook was written and checked without a GPU. Four things are unverified until its first run:
> 1. vLLM 0.30's Gated DeltaNet (FLA/Triton) and Triton attention kernels run Qwen3.8 on the T4's sm_75;
> 2. the 17.68 GiB of weights plus a 32k context fit 2× 15 GB at `--gpu-memory-utilization 0.90`;
> 3. `--tool-call-parser qwen3_coder` parses this quant's tool calls;
> 4. `--language-model-only` exists in vLLM 0.30.
>
> The startup probes in cell c confirm them: the server must start (1, 2, 4), then answer a structured-output request and a tool-call request (3). They run in the first ~30 min, before the long run. Troubleshooting: [evals/README.md](https://github.com/gfirik/agentic-analytics/blob/main/evals/README.md#troubleshooting).

**Before you run.** Open *Settings* and set **Accelerator: GPU T4 ×2** and **Internet: on**. Then use **Save Version → Save & Run All (Commit)**, which runs headless for up to 12 h and keeps everything in `/kaggle/working` as the version's output. Step-by-step instructions are in [evals/README.md](https://github.com/gfirik/agentic-analytics/blob/main/evals/README.md#run-the-full-suite-on-kaggle-free-2t4).

**Resumable.** The notebook runs one business per `aa eval --only <id>`, keeps every model response in a cache, and stops starting new businesses before the session budget runs out. If a session ends before all 30 are done, add this notebook's previous version as an input (*Add Input → Your Work → Notebooks*) and run again. It picks up the finished businesses and the cache from `/kaggle/input`, then continues. When all 30 are done, it merges them with `bun run eval:merge` (real timings) and zips everything into `/kaggle/working/aa-kaggle-results.zip`.

Cells: **0** settings · **a** GPUs · **b** install vLLM · **c** start and probe the server · **d** Node 22, Bun, clone, install · **e** point every role at vLLM · **f/h** run the suite in resumable chunks · **g** merge, collect, zip.

In [ ]:
# ── 0. Settings: change these, then Run All ────────────────────────────────────────────────────────────────────
# NOT YET RUN ON KAGGLE: vLLM 0.30's Gated DeltaNet/Triton attention kernels on sm_75, the 17.68 GiB weights + 32k
# context at 0.90, qwen3_coder on this quant's tool calls and --language-model-only in 0.30 are unverified; the startup
# probes in cell c confirm them (see the first cell).
import os, time
T0 = time.time()                      # session start: the time budget below counts from here

REPO_URL = "https://github.com/gfirik/agentic-analytics.git"
REPO_REF = "main"                     # a branch, tag or commit sha; pin a sha to reproduce a published row

# Model. Qwen publishes no official AWQ/INT4 of Qwen3.8-27B (only BF16, plus FP8, which T4 can't run natively).
# avyukth/Qwen3.8-27B-AWQ-INT4: W4A16, group 128, symmetric, compressed-tensors (llm-compressor 0.13); vision tower,
# lm_head and MTP head kept in BF16; 17.68 GiB of weights; its card serves it with vLLM. Alternative:
# mattbucci/Qwen3.8-27B-AWQ (W4A16 g128 converted to native AWQ GEMM format, ~18.7 GB, card tested on SGLang only).
#   https://huggingface.co/Qwen/Qwen3.8-27B   https://huggingface.co/avyukth/Qwen3.8-27B-AWQ-INT4
#   https://huggingface.co/mattbucci/Qwen3.8-27B-AWQ
MODEL_ID = "avyukth/Qwen3.8-27B-AWQ-INT4"
MODEL_REVISION = "5a2ee524cbc9f3ac5df12729ecf0c4b57f311192"   # the repo's commit (2026-08-15): the exact weights
SERVED_NAME = "Qwen3.8-27B-AWQ"       # the model id the engine sends; shows up in the results' routes
LABEL = "vllm (Kaggle 2×T4) Qwen3.8-27B-AWQ"   # the model row in evals/RESULTS.md and the README

# vLLM 0.30.0 (2026-09-22). Qwen3.8 reuses the Qwen3.5 architecture (hybrid Gated DeltaNet + gated attention), which
# vLLM has supported since 0.17; vLLM's Marlin W4A16 kernel needs compute capability 7.5 (T4 is 7.5) and its Exllama
# fallback 6.0 with fp16 activations. https://github.com/vllm-project/vllm/releases/tag/v0.30.0
# https://vllm.ai/blog/2026-08-12-qwen3.8 (same architecture as Qwen3.5, parsers qwen3_coder + qwen3)
VLLM_VERSION = "0.30.0"
MAX_MODEL_LEN = 32768       # see the note in cell c
GPU_MEM_UTIL = 0.90         # of each T4's 15 GB; lower it on OOM
MAX_NUM_SEQS = 8            # the engine sends one call at a time per job; a few in flight at most
ENFORCE_EAGER = False       # True skips CUDA graphs: slower decode, less memory, fewer T4 surprises
THINKING = False            # Qwen3.8 thinks by default; on 2×T4 that multiplies output tokens (see evals/README.md)

# Small-model mode halves the agent's tool budgets and findings (docs/specs/agent-and-verifier.md §8). It exists for
# small or rate-limited models; the Groq rows used it to fit a 200k-tokens/day quota. A 27B model with no rate limit
# is what normal mode is for, so it is off; set True for a faster run comparable with the Groq small-model rows.
SMALL_MODEL = False

NODE_VERSION = "22.23.3"    # checked against nodejs.org's SHASUMS256.txt for this version at download time

SESSION_BUDGET_H = 11.0     # Kaggle stops a session at 12 h; leave time to merge, zip and save the output
MODEL_TIMEOUT_MS = 900_000  # per request: a long answer at T4 decode speed takes minutes

WORK = "/kaggle/working"
OUT = f"{WORK}/aa-out"                  # everything worth keeping; carried forward between versions
CACHE = f"{OUT}/cache"                  # model response cache (AA_MODEL_CACHE): a rerun replays finished calls
CHUNKS = f"{OUT}/chunks"                # one results JSON per business (aa eval --only)
REPORTS = f"{OUT}/reports"              # each business's report.json (--keep-reports)
LOGS = f"{OUT}/logs"                    # model-calls.jsonl (the engine's call log), vLLM and eval logs
REPO = "/root/agentic-analytics"
VENV = "/root/vllm-env"
PORT = 8000
for d in (OUT, CACHE, CHUNKS, REPORTS, LOGS):
    os.makedirs(d, exist_ok=True)

import subprocess, sys
def sh(cmd, cwd=None, env=None, check=True, log=None):
    """Runs a bash command, streaming its output (and to `log`); returns (exit code, output)."""
    print(f"$ {cmd}", flush=True)
    p = subprocess.Popen(["bash", "-c", cmd], cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines = []
    with (open(log, "a") if log else open(os.devnull, "w")) as f:
        for line in p.stdout:
            print(line, end="", flush=True)
            f.write(line)
            lines.append(line)
    rc = p.wait()
    if check and rc != 0:
        raise RuntimeError(f"exit {rc}: {cmd}")
    return rc, "".join(lines)

## a. GPUs

Expect **2× Tesla T4**, about 15 GB each, compute capability 7.5. A T4 has **no bfloat16**, so vLLM runs in `--dtype half`. The driver's CUDA version picks the vLLM wheel in the next cell.

In [ ]:
import re
_, smi = sh("nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv,noheader")
gpus = [l for l in smi.strip().splitlines() if l.strip()]
sh("nvidia-smi")
_, full = sh("nvidia-smi | head -n 4", check=False)
m = re.search(r"CUDA Version:\s*(\d+)\.(\d+)", full)
DRIVER_CUDA = (int(m.group(1)), int(m.group(2))) if m else (0, 0)
print(f"\n{len(gpus)} GPU(s); the driver supports CUDA up to {DRIVER_CUDA[0]}.{DRIVER_CUDA[1]}")
assert len(gpus) == 2 and all("T4" in g for g in gpus), "Set Settings → Accelerator → GPU T4 ×2"

## b. Install vLLM

vLLM goes in its own virtual environment (made with `uv`), so it can't clash with Kaggle's preinstalled packages.

- **Wheel.** The default PyPI wheel of vLLM 0.30.0 is built for **CUDA 13.0**, which needs a driver that reports CUDA ≥ 13.0. On an older driver, the cell installs the release's **+cu129** wheel with the matching PyTorch instead ([release notes](https://github.com/vllm-project/vllm/releases/tag/v0.30.0); [wheel naming](https://pydevtools.com/handbook/how-to/how-to-use-vllm-with-uv/)).
- **Weights.** The cell then downloads the weights (about 18 GB, Apache-2.0, not gated). An `HF_TOKEN` Kaggle secret is used if you add one: it's optional and only raises Hugging Face's download rate limit.

In [ ]:
sh(f"{sys.executable} -m pip install -q uv")
UV = f"{sys.executable} -m uv"
if not os.path.exists(f"{VENV}/bin/python"):
    sh(f"{UV} venv --python 3.12 {VENV}")
if DRIVER_CUDA >= (13, 0):
    sh(f"{UV} pip install --python {VENV}/bin/python vllm=={VLLM_VERSION} --torch-backend=auto")
else:
    # Not hash-pinned: the release publishes no checksum list for its wheels; it comes from the vllm-project release
    # over HTTPS, and the version is pinned.
    wheel = f"https://github.com/vllm-project/vllm/releases/download/v{VLLM_VERSION}/vllm-{VLLM_VERSION}+cu129-cp38-abi3-manylinux_2_34_x86_64.whl"
    sh(f"{UV} pip install --python {VENV}/bin/python '{wheel}' --torch-backend=cu129")
sh(f"{VENV}/bin/python -c 'import vllm, torch; print(\"vllm\", vllm.__version__, \"torch\", torch.__version__, \"cuda\", torch.version.cuda)'")

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN: from Kaggle secrets")
except Exception:
    print("HF_TOKEN: none (fine: the model is not gated)")
t = time.time()
sh(f"{VENV}/bin/python -c 'from huggingface_hub import snapshot_download; print(snapshot_download(\"{MODEL_ID}\", revision=\"{MODEL_REVISION}\"))'")
print(f"weights ready in {(time.time() - t) / 60:.1f} min")

## c. Start the vLLM server and probe it

**Flags, and why:**

- `--tensor-parallel-size 2` puts half the weights on each T4 (about 8.5 GiB each). `--dtype half` is needed because a T4 has no bf16.
- `--disable-custom-all-reduce`: the two T4s talk over PCIe without peer-to-peer access, so NCCL does the all-reduce.
- `--language-model-only` skips the vision tower, which the engine never uses.
- `--max-model-len 32768`. The engine's largest request so far was a DECIDE prompt of more than 7,000 input tokens (printing_shop-v1, small-model mode on Groq). In normal mode, the agent's tool loop grows its message history to at most 25 tool calls. 32k leaves room for that plus the answer. The KV cost is small, because only 16 of the 64 layers are full attention (4 KV heads × 256 dims): about 32 KiB per token per GPU, so about 1 GiB at 32k. The other layers keep a fixed-size Gated DeltaNet state.
- `--enable-auto-tool-choice --tool-call-parser qwen3_coder`: the engine's tool loop (`generateWithTools` in `packages/models/src/ai-sdk.ts`) sends OpenAI `tools` through the Vercel AI SDK's OpenAI-compatible provider, and the Qwen3.5/3.8 recipes use this parser.
- `--reasoning-parser qwen3` keeps any thinking out of `content`, so JSON answers parse. With `THINKING = False`, `--default-chat-template-kwargs '{"enable_thinking": false}'` turns thinking off for every request. The engine never sends `chat_template_kwargs` itself.
- **Structured output.** A structured-output call (`generateObject`) carries no tools and asks for `response_format: json_schema`, because the `vllm` endpoint in cell e sets `structured_outputs: true`. vLLM enforces the schema with guided decoding.
- **Sampling.** `--override-generation-config` uses Qwen's non-thinking settings (temperature 0.7, top_p 0.8, top_k 20) without the presence penalty, which would penalise the repeated keys of JSON. The engine sends no sampling parameters.

**The probes.** The cell waits for `/v1/models`, then sends one structured-output request and one tool-call request, the two kinds of call the engine makes. That way a parser or kernel problem shows up now, not hours into the run.

In [ ]:
import json, urllib.request
vllm_log = f"{LOGS}/vllm.log"
flags = [
    f"--revision {MODEL_REVISION}", f"--served-model-name {SERVED_NAME}", f"--host 127.0.0.1 --port {PORT}",
    "--tensor-parallel-size 2", "--dtype half", "--disable-custom-all-reduce", "--language-model-only",
    f"--max-model-len {MAX_MODEL_LEN}", f"--gpu-memory-utilization {GPU_MEM_UTIL}", f"--max-num-seqs {MAX_NUM_SEQS}",
    "--enable-auto-tool-choice --tool-call-parser qwen3_coder", "--reasoning-parser qwen3",
    """--override-generation-config '{"temperature": 0.7, "top_p": 0.8, "top_k": 20}'""",
]
if not THINKING:
    flags.append("""--default-chat-template-kwargs '{"enable_thinking": false}'""")
if ENFORCE_EAGER:
    flags.append("--enforce-eager")
cmd = f"{VENV}/bin/vllm serve {MODEL_ID} " + " ".join(flags)
print(cmd)
server = subprocess.Popen(["bash", "-c", f"exec {cmd} >> {vllm_log} 2>&1"], env={**os.environ, "VLLM_WORKER_MULTIPROC_METHOD": "spawn"})

BASE = f"http://127.0.0.1:{PORT}/v1"
def api(path, body=None, timeout=600):
    req = urllib.request.Request(BASE + path, data=None if body is None else json.dumps(body).encode(), headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return json.loads(r.read())

def server_up():
    try:
        return SERVED_NAME in json.dumps(api("/models", timeout=5))
    except Exception:
        return False

t = time.time()
while not server_up():
    if server.poll() is not None:
        sh(f"tail -n 80 {vllm_log}", check=False)
        raise RuntimeError("vLLM exited; see the log above and Troubleshooting in evals/README.md")
    if time.time() - t > 45 * 60:
        raise RuntimeError("vLLM not up after 45 min")
    time.sleep(15)
print(f"vLLM up in {(time.time() - t) / 60:.1f} min")
sh(f"grep -iE 'backend|kernel|KV cache|maximum concurrency' {vllm_log} | tail -n 20", check=False)

# Probe 1: structured output, as generateObject sends it.
schema = {"type": "object", "properties": {"metric": {"type": "string"}, "change_pct": {"type": "number"}}, "required": ["metric", "change_pct"], "additionalProperties": False}
t = time.time()
r = api("/chat/completions", {"model": SERVED_NAME, "messages": [{"role": "user", "content": "Revenue fell from 200 to 150. Answer as JSON with the metric name and the change in percent."}],
                              "response_format": {"type": "json_schema", "json_schema": {"name": "probe", "schema": schema, "strict": True}}})
answer = json.loads(r["choices"][0]["message"]["content"])
print("structured output OK:", answer, f"({r['usage']['completion_tokens']} tokens in {time.time() - t:.1f} s)")

# Probe 2: a tool call, as generateWithTools sends it.
tools = [{"type": "function", "function": {"name": "run_query", "description": "Run a named analysis query", "parameters": {"type": "object", "properties": {"metric": {"type": "string"}, "period": {"type": "string"}}, "required": ["metric", "period"]}}}]
r = api("/chat/completions", {"model": SERVED_NAME, "tool_choice": "auto", "tools": tools, "messages": [{"role": "user", "content": "Use the tool to get revenue for 2026-04."}]})
calls = r["choices"][0]["message"].get("tool_calls") or []
assert calls and calls[0]["function"]["name"] == "run_query", f"no tool call parsed: {r['choices'][0]['message']}"
print("tool call OK:", calls[0]["function"]["name"], calls[0]["function"]["arguments"])

## d. Node 22, Bun, clone, install

The engine runs on Node 22. Bun is only the package manager, pinned to the repo's `packageManager` version. The Node tarball is checked against the official SHA-256 list.

In [ ]:
import hashlib, re, tarfile, urllib.request
base = f"https://nodejs.org/dist/v{NODE_VERSION}/"
name = f"node-v{NODE_VERSION}-linux-x64.tar.xz"
sums = urllib.request.urlopen(base + "SHASUMS256.txt").read().decode()
digest = re.search(rf"^([0-9a-f]{{64}})\s+{re.escape(name)}$", sums, re.M).group(1)
tarball = f"/root/{name}"
urllib.request.urlretrieve(base + name, tarball)
assert hashlib.sha256(open(tarball, "rb").read()).hexdigest() == digest, "Node tarball checksum mismatch"
with tarfile.open(tarball) as t:
    t.extractall("/opt")
NODE_BIN = f"/opt/{name.removesuffix('.tar.xz')}/bin"
os.environ["PATH"] = f"{NODE_BIN}:{os.environ['PATH']}"
sh("node --version")

if not os.path.isdir(REPO):
    sh(f"git clone {REPO_URL} {REPO}")
sh(f"git fetch --depth 1 origin {REPO_REF} && git checkout --detach FETCH_HEAD", cwd=REPO)
_, COMMIT = sh("git rev-parse HEAD", cwd=REPO)
COMMIT = COMMIT.strip()
bun_version = json.load(open(f"{REPO}/package.json"))["packageManager"].split("@")[1]
sh(f"npm install -g bun@{bun_version}")
sh("bun --version && bun install --frozen-lockfile", cwd=REPO)

## e. Point every role at the local vLLM

- **Endpoint.** `agentic-analytics.config.yaml` in the clone adds an OpenAI-compatible endpoint named `vllm` at `http://127.0.0.1:8000/v1`. It sets `structured_outputs: true` and no `api_key_env`, because vLLM runs without a key here (docs/specs/models.md §4).
- **Roles.** `PLANNER_MODEL`, `EXTRACTOR_MODEL` and `JUDGE_MODEL` all name `openai-compatible:vllm:<served name>`, and `--model configured` uses them.
- **Embeddings.** They stay in-process. `aa eval` uses the deterministic hash embedder, like every other row in RESULTS.md.

In [ ]:
with open(f"{REPO}/agentic-analytics.config.yaml", "w") as f:
    f.write(f"""# Written by evals/kaggle/full-suite-vllm.ipynb: the vLLM server in this notebook.
providers:
  openai-compatible:
    vllm: {{ base_url: "http://127.0.0.1:{PORT}/v1", structured_outputs: true }}
""")
ROUTE = f"openai-compatible:vllm:{SERVED_NAME}"
ENV = {**os.environ,
       "PLANNER_MODEL": ROUTE, "EXTRACTOR_MODEL": ROUTE, "JUDGE_MODEL": ROUTE,
       "MODEL_TIMEOUT_MS": str(MODEL_TIMEOUT_MS),
       "AA_MODEL_CACHE": CACHE,
       "LOG_LEVEL": "warn"}
# The engine appends every call to data/logs/model-calls.jsonl (role, route, status, tokens; no prompt text):
# keep it in the output folder so it survives the session.
os.makedirs(f"{REPO}/data", exist_ok=True)
if not os.path.islink(f"{REPO}/data/logs"):
    sh(f"rm -rf {REPO}/data/logs && ln -s {LOGS} {REPO}/data/logs")
sh("bun run aa doctor", cwd=REPO, env=ENV, check=False)

## f/h. Run the suite, one business at a time, resumable

**Expected runtime.** These are estimates, not yet measured on a T4 (the first businesses print the real figure).

- **Speed.** Decode on 2×T4 with a 4-bit 27B model runs at roughly 10–20 tokens/s. A Groq small-model job used about 25k input and 6k output tokens. Normal mode makes more tool calls and writes more findings, so expect about 10–15 min per business.
- **Totals.** Normal mode takes about **5–8 h** for 30 businesses, plus about 20 min of setup. Small-model mode is roughly half that.
- **Thinking.** With `THINKING = True`, expect several times longer. That doesn't fit one session: it runs over several sessions through the resume below.

**Resume.** Each business is its own `bun run eval -- --suite full --model configured --only <id>`, and its results JSON lands in `aa-out/chunks/<id>/`.

- **Budget.** Before starting a business, the cell checks that the session budget leaves 1.5× the slowest business so far. If it doesn't, it stops cleanly.
- **Carried forward.** Finished businesses and the response cache are copied in from any earlier version of this notebook attached under `/kaggle/input`.
- **Retries.** A business that failed on the server (connection or 5xx) is retried once.

In [ ]:
import glob, shutil
# Carry forward an earlier version's output (Add Input → Your Work → Notebooks → this notebook).
for prev in glob.glob("/kaggle/input/**/aa-out", recursive=True):
    print("resuming from", prev)
    shutil.copytree(prev, OUT, dirs_exist_ok=True)

DATASETS = sorted(d for d in os.listdir(f"{REPO}/evals/datasets/generated") if re.fullmatch(r"[a-z_]+-v\d", d))
assert len(DATASETS) == 30, DATASETS

def chunk_result(ds):
    files = glob.glob(f"{CHUNKS}/{ds}/*-full.json")
    return json.load(open(files[0])) if files else None

# Retry only what the server caused: a lost connection or an HTTP 5xx. A PROVIDER error such as "maximum context
# length" is the model's result and is kept.
INFRA = re.compile(r"ECONNREFUSED|ECONNRESET|fetch failed|Cannot connect|socket hang up|other side closed|\b5\d\d\b")
def infra_failure(r):
    s = r["scores"][0]
    return not s["completed"] and bool(INFRA.search(s.get("error", "")))

small_flag = " --small-model" if SMALL_MODEL else ""
durations = [chunk_result(d)["scores"][0]["seconds"] for d in DATASETS if chunk_result(d)]
todo = [d for d in DATASETS if not chunk_result(d) or infra_failure(chunk_result(d))]
print(f"{30 - len(todo)} of 30 businesses already done; {len(todo)} to run")
retried = set()
SERVER_DOWN = False
while todo:
    ds = todo[0]
    left = SESSION_BUDGET_H * 3600 - (time.time() - T0)
    need = 1.5 * max(durations, default=30 * 60)
    if left < need:
        print(f"stopping: {left / 60:.0f} min left in the session budget, a business may take {need / 60:.0f} min")
        break
    if not server_up():
        # Keep what finished: the next cell still zips the output, then stops with this error.
        SERVER_DOWN = True
        print("vLLM is down; see aa-out/logs/vllm.log. Stopping here; the next cell saves the partial output.")
        break
    shutil.rmtree(f"{CHUNKS}/{ds}", ignore_errors=True)
    t = time.time()
    sh(f"bun run eval -- --suite full --model configured --only {ds} --results-dir {CHUNKS}/{ds} --keep-reports {REPORTS}{small_flag}",
       cwd=REPO, env=ENV, check=False, log=f"{LOGS}/eval.log")   # exits 1 when a bar fails: the JSON is what counts
    r = chunk_result(ds)
    if r is None:
        raise RuntimeError(f"{ds}: no results file; see aa-out/logs/eval.log")
    if infra_failure(r) and ds not in retried:
        retried.add(ds)
        print(f"{ds}: server-side failure ({r['scores'][0]['error'][:120]}); retrying once")
        continue
    todo.pop(0)
    durations.append(time.time() - t)
    done = 30 - len(todo)
    avg = sum(durations) / len(durations)
    print(f"── {done}/30 done · this one {durations[-1] / 60:.1f} min · average {avg / 60:.1f} min · about {avg * len(todo) / 3600:.1f} h to go\n")
ALL_DONE = all(chunk_result(d) for d in DATASETS)
print("all 30 businesses done" if ALL_DONE else f"{sum(1 for d in DATASETS if chunk_result(d))}/30 done: save this version, attach its output as an input to the next one and run again")

## g. Merge, collect and zip

- **Merge.** `bun run eval:merge` joins the 30 one-business results into one results file, in suite order with real timings, and the bars are recomputed over all 30. The file is `evals/results/<date>-vllm-kaggle-2-t4-qwen3.8-27b-awq-full.json` (`…-awq-small-full.json` in small-model mode). The cell also regenerates `evals/RESULTS.md`.
- **Zip.** The zip holds that JSON, RESULTS.md, the engine's call log, every report.json, the per-business results, the vLLM and eval logs, and the settings.
- **Publish.** Download the zip, copy the JSON into `evals/results/` of your clone, then run `bun run readme:assets` and open a PR (evals/README.md).

In [ ]:
mode = "small-model mode" if SMALL_MODEL else "normal mode"
NOTE = (f"Self-hosted: vLLM {VLLM_VERSION} in a Kaggle notebook (evals/kaggle/full-suite-vllm.ipynb) on 2× Tesla T4, fp16, "
        f"tensor parallel 2, max context {MAX_MODEL_LEN}. Model {MODEL_ID}, a community 4-bit (W4A16) quant of "
        f"Qwen/Qwen3.8-27B (revision {MODEL_REVISION[:12]}), for every role (planner, extractor, judge); thinking {'on' if THINKING else 'off'}; {mode}; "
        f"hash embedder like every other row. One business per `aa eval --only`, merged with `bun run eval:merge` "
        f"(real timings). Engine commit {COMMIT[:12]}.")
with open(f"{OUT}/settings.json", "w") as f:
    json.dump({k: globals()[k] for k in ["REPO_URL", "REPO_REF", "COMMIT", "MODEL_ID", "MODEL_REVISION", "NODE_VERSION", "SERVED_NAME", "LABEL", "VLLM_VERSION", "MAX_MODEL_LEN", "GPU_MEM_UTIL", "MAX_NUM_SEQS", "ENFORCE_EAGER", "THINKING", "SMALL_MODEL", "MODEL_TIMEOUT_MS"]}, f, indent=2)

if ALL_DONE:
    parts = " ".join(sorted(glob.glob(f"{CHUNKS}/*/*-full.json")))
    note = NOTE.replace("'", "'\\''")
    sh(f"bun run eval:merge -- --label '{LABEL}' --note '{note}'{small_flag} {parts}", cwd=REPO, env=ENV)
    # The file name aa eval gives a label (evalSlug in evals/runner/src/cli.ts).
    slug = re.sub(r"[^a-z0-9.]+", "-", LABEL.lower()).strip("-")
    published = sorted(glob.glob(f"{REPO}/evals/results/*-{slug}{'-small' if SMALL_MODEL else ''}-full.json"))[-1]
    shutil.copy(published, OUT)
    shutil.copy(f"{REPO}/evals/RESULTS.md", OUT)
    print("results:", os.path.basename(published))
    sh(f"grep -A 16 '### {LABEL}' {REPO}/evals/RESULTS.md | head -n 20", check=False)
shutil.make_archive(f"{WORK}/aa-kaggle-results", "zip", OUT)
print(f"\n{WORK}/aa-kaggle-results.zip ({os.path.getsize(f'{WORK}/aa-kaggle-results.zip') / 1e6:.1f} MB)")
print("complete: download the zip" if ALL_DONE else "partial: run the next version with this one's output as an input")
if SERVER_DOWN:
    raise RuntimeError("vLLM went down during the run (aa-out/logs/vllm.log); the partial output above is saved")

In [ ]:
# Stop the server (a committed run ends here anyway).
server.terminate()